# Week 8: Advanced Text Generation Techniques and Tools

Last week we wrote one prompt and got one answer. Today we wrap the model in a little Python so it can do more:

1. **Chains:** the output of one prompt becomes the input of the next.
2. **Memory:** the model forgets everything between calls, so we paste the past back in.

The big idea: **both are just ways of building the prompt.** Frameworks like LangChain package them, however, we'll do this manually in this lecture.

**Running scenario:** AeroBeat, a small headphone shop. The review from Weeks 5 and 7, *"not bad at all, actually pretty good"*, is one of its reviews.

## 0. Setup

Same model as Week 7, `Qwen2.5-1.5B-Instruct`.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()
model.generation_config.do_sample = False          # greedy: same output for everyone
model.generation_config.temperature = model.generation_config.top_p = model.generation_config.top_k = None

# Slides 4-5: memory = number of weights x bits per weight / 8
n = sum(p.numel() for p in model.parameters())
for name, bits in [("float32", 32), ("float16", 16), ("4-bit", 4)]:
    print(f"{name:8s} {n * bits / 8 / 1e9:4.2f} GB")

Loading weights: 100%|██████████| 338/338 [00:13<00:00, 24.59it/s]


float32  6.17 GB
float16  3.09 GB
4-bit    0.77 GB


4-bit quantization shrinks the model to a quarter of float16.

Two helpers used all class: `generate()` sends chat messages to the model, and `count_tokens()` measures how long the prompt is.

In [ ]:
def to_prompt(messages):                     # the chat template text from Week 7
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate(messages, max_new_tokens=100):
    ids = tokenizer(to_prompt(messages), return_tensors="pt", add_special_tokens=False).to(device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def count_tokens(messages):
    return len(tokenizer(to_prompt(messages), add_special_tokens=False)["input_ids"])

## 1. Chains

A **prompt template** is a prompt with blanks. A chain fills the blanks and calls the model. Link two of them and step 1's output fills a blank in step 2.

In [ ]:
SUPPORT = "You are a customer support assistant for AeroBeat, an online headphone shop."

def make_chain(template, max_new_tokens=80):
    return lambda **inputs: generate([{"role": "system", "content": SUPPORT},
                                      {"role": "user", "content": template.format(**inputs)}], max_new_tokens)

label_chain = make_chain("Review: {review}\nIs this review positive or negative? Answer with one word.", 3)
reply_chain = make_chain("Review: {review}\nThe review is {label}. Write a two-sentence reply to the customer.")

review = "not bad at all, actually pretty good"
label = label_chain(review=review).strip().lower().rstrip(".")   # step 1
reply = reply_chain(review=review, label=label)                   # step 2 uses step 1
print("LABEL:", label, "\nREPLY:", reply)

LABEL: positive 
REPLY: Thank you for your feedback! We're glad to hear that you found our headphones satisfactory and even appreciated their quality. If there's anything else we can improve upon, please don't hesitate to let us know.


Because `label` is a Python variable, the shop's code can act on it (for example, send negative reviews to a human). One big prompt asking for "label and reply" would return one blob of text. The cost of a chain: 2 model calls instead of 1.

## 2. Memory

The model is **stateless**: each call sees only the messages we send in that call. Sara, who wrote our review, chats with the shop. She gives her name in turn 1 and asks for it in turn 4.

Three memory types. The only line that differs is `past`, which decides which old messages we paste back in:

* **none:** nothing
* **buffer:** the whole history
* **window:** only the last exchange (one user message and one reply)

In [ ]:
TURNS = ["Hi, I'm Sara. I bought the AeroBeat X1 headphones last week.",
         "How should I clean the ear cushions?",
         "Any tips to make the battery last longer?",
         "Thanks! What name did I give you at the start of our chat?"]
SYSTEM = {"role": "system", "content": SUPPORT + " Use anything the customer told you earlier "
                                       "in this conversation. Answer in at most two sentences."}

def run_chat(memory):
    history = []                                      # every past message, user and assistant
    for user_msg in TURNS:
        if memory == "none":     past = []
        elif memory == "buffer": past = history
        elif memory == "window": past = history[-2:]  # last exchange only (1 user + 1 assistant)
        messages = [SYSTEM] + past + [{"role": "user", "content": user_msg}]
        reply = generate(messages, 60)
        history += [{"role": "user", "content": user_msg},
                    {"role": "assistant", "content": reply}]
        print(f"{count_tokens(messages):4d} tokens | {reply[:80]}")

for memory in ["none", "buffer", "window"]:
    print(f"\n--- {memory} ---")
    run_chat(memory)


--- none ---
  63 tokens | Hello Sara! Thank you for choosing AeroBeat. How can we assist you with your pur
  54 tokens | To clean the ear cushions of your headphones, gently wipe them with a soft cloth
  55 tokens | To help your headphones last longer on battery power, try reducing volume levels
  61 tokens | At the start of our chat, you mentioned that your name is John.

--- buffer ---
  63 tokens | Hello Sara! Thank you for choosing AeroBeat. How can we assist you with your pur
 100 tokens | To clean the ear cushions of your AeroBeat X1 headphones, gently wipe them with 
 159 tokens | Yes, there are several ways to help extend the battery life of your AeroBeat X1 
 244 tokens | At the beginning of our chat, you introduced yourself as Sara.

--- window ---
  63 tokens | Hello Sara! Thank you for choosing AeroBeat. How can we assist you with your pur
 100 tokens | To clean the ear cushions of your AeroBeat X1 headphones, gently wipe them with 
 113 tokens | Yes, there are several wa

Read turn 4 of each run. In our run:

| Memory | Prompt at turn 4 | Turn 4 answer | Why |
|---|---|---|---|
| none | 61 tokens | "your name is **John**" | Sara's introduction is not in the prompt. The model does not just forget, it confidently makes a name up (a **hallucination**). |
| buffer | 244 tokens | "you introduced yourself as **Sara**" | The whole chat is pasted back in, so the name is there. But the prompt grew about 4x in 4 turns and keeps growing until it hits the context window (Week 3). |
| window | 139 tokens | "I don't have access to..." | The prompt stays small, but turn 1 has slid out of the window, so the name is gone. |

No memory type is free: you trade prompt length against what the model can still see. 

## Summary memory

Build the summary memory using the `TURNS`, `SYSTEM` and `generate()`.

1. Write `update_summary(summary, user_msg, reply)`. It calls the model with the current summary and the newest exchange and returns an updated summary of at most three sentences. Ask it to keep names and products.
2. Write `run_summary_chat()`. Each turn sends only the system message (with the summary added to it) and the new user message, and prints the prompt's token count and the reply as in `run_chat()`. After each turn, update the summary.
3. Answer:
   * Does the assistant remember Sara at turn 4? Print the final summary.
   * How does the prompt size at turn 4 compare with the buffer and window runs?
   * How many model calls did the whole chat take, compared with the buffer run?
   * What could go wrong with this memory in a long conversation?

In [ ]:
def update_summary(summary, user_msg, reply):
    messages = [
        {"role": "system", "content": "You summarize conversations. Keep names, products and key facts. "
                                      "At most three sentences."},
        {"role": "user", "content": f"Current summary: {summary or '(empty)'}\n\n"
                                    f"New exchange:\nCustomer: {user_msg}\nAssistant: {reply}\n\n"
                                    "Write the updated summary."}]
    return generate(messages, 100)

def run_summary_chat():
    summary, calls = "", 0
    for user_msg in TURNS:
        system = {"role": "system",
                  "content": SYSTEM["content"] + (f"\nSummary of the conversation so far: {summary}" if summary else "")}
        messages = [system, {"role": "user", "content": user_msg}]
        reply = generate(messages, 60); calls += 1
        summary = update_summary(summary, user_msg, reply); calls += 1
        print(f"{count_tokens(messages):4d} tokens | {reply[:80]}")
    print(f"\nModel calls: {calls}\nFinal summary: {summary}")

run_summary_chat()

  63 tokens | Hello Sara! Thank you for choosing AeroBeat. How can we assist you with your pur
  92 tokens | To clean the ear cushions of your AeroBeat X1 headphones, gently wipe them with 
 106 tokens | To help prolong the life of your AeroBeat X1 headphones' battery, it's recommend
 113 tokens | At the start of our chat, you mentioned that your name was Sara.

Model calls: 8
Final summary: Sara is seeking tips on prolonging the life of her AeroBeat X1 headphones' battery. The assistant suggests avoiding overcharging and maintaining the batteries in a clean, dry environment to enhance longevity.
